In [1]:
import os, sys
import math
from copy import deepcopy
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

import torch as th
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
os.environ['DGLBACKEND'] = 'pytorch'
import dgl
from dgl.data import DGLDataset
from dgl.dataloading import GraphDataLoader
from torch.utils.data import Subset
from torch.utils.data.sampler import SubsetRandomSampler

import dgl.function as fn

/Users/nad/miniconda3/envs/dgl_env/lib/python3.12/site-packages/torchdata/datapipes/__init__.py:18: UserWarning: 
################################################################################
WARNING!
The 'datapipes', 'dataloader2' modules are deprecated and will be removed in a
future torchdata release! Please see https://github.com/pytorch/data/issues/1196
to learn more and leave feedback.
################################################################################

  deprecation_warning()
/Users/nad/miniconda3/envs/dgl_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from scripts.n15_create_dgl_dataset import GraphsFromCSVDataset
import config

In [13]:
HIERARCHY_ROOTS = [
    "",
    "Class I (Retrotransposons)",
    "Class II (DNA transposons)",
    "Class I (Retrotransposons)\tLTR Retrotransposon",
    "Class I (Retrotransposons)\tNon-LTR Retrotransposon",
]

In [44]:

datasets_plants = []

for root in HIERARCHY_ROOTS:
    dataset = GraphsFromCSVDataset(
        nodes_csv="/Users/nad/mobiraph/data/n33_dotplots/params_20_16_plant_new/nodes.csv",
        edges_csv="/Users/nad/mobiraph/data/n33_dotplots/params_20_16_plant_new/edges.csv",
        metadata_json="/Users/nad/mobiraph/data/n26_sv_processed/hierarchy_sequences_sv_plants.json",
        hierarchy_root=root,
        save_dir=f"/Users/nad/mobiraph/data/n33_dotplots/dgl_dataset_plants_{root}",
        force_reload=False,
        verbose=True,
    )
    print(dataset.label2id)

Done loading data from cached files.
{'BEL': 0, 'Copia': 1, 'DIRS': 2, 'Gypsy': 3, 'Troyka': 4}
Done loading data from cached files.
{'CR1': 0, 'L1': 1, 'Non-LTR Retrotransposon_other': 2, 'RTE': 3, 'RTEX': 4, 'Tad1': 5, 'Tx1': 6}


In [7]:

datasets_repbase = []

for root in HIERARCHY_ROOTS:
    dataset = GraphsFromCSVDataset(
        nodes_csv="/Users/nad/mobiraph/data/n33_dotplots/all/nodes.csv",
        edges_csv="/Users/nad/mobiraph/data/n33_dotplots/all/edges.csv",
        metadata_json="/Users/nad/mobiraph/data/n13_repbase_processed/hierarchy_sequences_02_ltr_correction_with_classes.json",
        hierarchy_root=root,
        save_dir=f"/Users/nad/mobiraph/data/n33_dotplots/dgl_dataset_repbase_{root}",
        force_reload=True,
        verbose=True,
    )

Done saving data into cached files.


In [42]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl

from torch.utils.data import DataLoader, Subset
from dgl.nn import GATConv, AvgPooling, MaxPooling


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def collate_fn(samples):
    graph_ids, graphs, labels = map(list, zip(*samples))
    bg = dgl.batch(graphs)
    labels = torch.tensor(labels)
    return graph_ids, bg, labels


class GATGraphClassifier(nn.Module):
    def __init__(
        self,
        in_feats,
        hidden_feats,
        num_classes,
        num_heads=4,
        dropout=0.2,
        use_max_pool=True,
    ):
        super().__init__()

        self.gat1 = GATConv(
            in_feats=in_feats,
            out_feats=hidden_feats,
            num_heads=num_heads,
            feat_drop=dropout,
            attn_drop=dropout,
            allow_zero_in_degree=True,
        )

        self.gat2 = GATConv(
            in_feats=hidden_feats,
            out_feats=hidden_feats,
            num_heads=num_heads,
            feat_drop=dropout,
            attn_drop=dropout,
            allow_zero_in_degree=True,
        )

        self.avg_pool = AvgPooling()
        self.max_pool = MaxPooling() if use_max_pool else None

        readout_dim = hidden_feats * 2 if use_max_pool else hidden_feats

        self.classifier = nn.Sequential(
            nn.Linear(readout_dim, hidden_feats),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_feats, num_classes),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        x = g.ndata["feat"].float()

        # [N, num_heads, hidden_feats]
        x = self.gat1(g, x)
        x = x.mean(dim=1)  # -> [N, hidden_feats]
        x = F.relu(x)
        x = self.dropout(x)

        # [N, num_heads, hidden_feats]
        x = self.gat2(g, x)
        x = x.mean(dim=1)  # -> [N, hidden_feats]
        x = F.relu(x)

        g.ndata["h"] = x

        h_avg = self.avg_pool(g, x)

        if self.max_pool is not None:
            h_max = self.max_pool(g, x)
            hg = torch.cat([h_avg, h_max], dim=1)
        else:
            hg = h_avg

        logits = self.classifier(hg)
        return logits


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    criterion = nn.CrossEntropyLoss()

    for graph_ids, bg, labels in dataloader:
        bg = bg.to(device)
        labels = labels.to(device)

        logits = model(bg)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples
    return avg_loss, acc



def train_model(
    dataset,
    split_idx,
    hidden_feats=64,
    num_heads=4,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    num_epochs=50,
    dropout=0.2,
    device=None,
    use_max_pool=True,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    train_dataset = Subset(dataset, split_idx["train"].tolist())
    val_dataset = Subset(dataset, split_idx["valid"].tolist())
    test_dataset = Subset(dataset, split_idx["test"].tolist())

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    _, sample_graph, _ = dataset[0]
    in_feats = sample_graph.ndata["feat"].shape[1]
    num_classes = len(dataset.label2id)

    model = GATGraphClassifier(
        in_feats=in_feats,
        hidden_feats=hidden_feats,
        num_classes=num_classes,
        num_heads=num_heads,
        dropout=dropout,
        use_max_pool=use_max_pool,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state = None

    for epoch in range(1, num_epochs + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for graph_ids, bg, labels in train_loader:
            bg = bg.to(device)
            labels = labels.to(device)

            logits = model(bg)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

        train_loss = total_loss / total_samples
        train_acc = total_correct / total_samples

        val_loss, val_acc = evaluate(model, val_loader, device)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = evaluate(model, test_loader, device)
    print(f"\nBest val_acc={best_val_acc:.4f}")
    print(f"Test loss={test_loss:.4f}, test_acc={test_acc:.4f}")

    return model

In [43]:
models = {}
for root in HIERARCHY_ROOTS:
    dataset = GraphsFromCSVDataset(
        nodes_csv="/Users/nad/mobiraph/data/n33_dotplots/params_20_16_plant_new/nodes.csv",
        edges_csv="/Users/nad/mobiraph/data/n33_dotplots/params_20_16_plant_new/edges.csv",
        metadata_json="/Users/nad/mobiraph/data/n26_sv_processed/hierarchy_sequences_sv_plants.json",
        hierarchy_root=root,
        save_dir=f"/Users/nad/mobiraph/data/n33_dotplots/dgl_dataset_plants_{root}",
        force_reload=False,
        verbose=True,
    )
    print("Число графов:", len(dataset))
    print("Классы:", dataset.label2id)

    split_idx = dataset.split_idx(
        train_ratio=0.9,
        val_ratio=0.05,
        test_ratio=0.05,
        seed=42,
        stratified=True,
    )

    model = train_model(
        dataset=dataset,
        split_idx=split_idx,
        hidden_feats=64,
        batch_size=32,
        lr=1e-3,
        weight_decay=1e-4,
        num_epochs=10,
        dropout=0.2,
        use_max_pool=False
    )
    models[root] = model
    if root == "":
        save_dir = Path("/Users/nad/mobiraph/data/n33_dotplots/models/root")
    else:
        save_dir = Path(f"/Users/nad/mobiraph/data/n33_dotplots/models/{root}")

    save_dir.mkdir(parents=True, exist_ok=True)

    save_path = save_dir / "gat_model.pt"
    torch.save(model.state_dict(), save_path)

    print(f"Model saved to {save_path}")

KeyboardInterrupt: 

In [4]:
test_dataset = GraphsFromCSVDataset(
        nodes_csv="/Users/nad/mobiraph/data/n33_dotplots/all/nodes.csv",
        edges_csv="/Users/nad/mobiraph/data/n33_dotplots/all/edges.csv",
        metadata_json="/Users/nad/mobiraph/data/n13_repbase_processed/hierarchy_sequences_02_ltr_correction_with_classes.json",
        hierarchy_root='',
        save_dir=f"/Users/nad/mobiraph/data/n33_dotplots/dgl_dataset_repbase_",
        force_reload=True,
        verbose=True,
    )

Done saving data into cached files.


In [5]:
len(test_dataset)

69472

In [29]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
)

In [30]:
import torch
import pandas as pd

@torch.no_grad()
def predict_with_logits(model, dataloader, device):
    model.eval()

    all_graph_ids = []
    all_logits = []
    all_preds = []
    all_labels = []

    for graph_ids, bg, labels in dataloader:
        bg = bg.to(device)

        logits = model(bg)
        preds = logits.argmax(dim=1).cpu()

        all_graph_ids.extend(graph_ids)
        all_logits.append(logits.cpu())
        all_preds.append(preds)
        all_labels.append(labels)

    return (
        all_graph_ids,
        torch.cat(all_logits, dim=0),
        torch.cat(all_preds, dim=0),
        torch.cat(all_labels, dim=0),
    )

In [31]:
test_dataset[0]

('AACOPIA1_I',
 Graph(num_nodes=24, num_edges=14,
       ndata_schemes={'feat': Scheme(shape=(5,), dtype=torch.float32), 'node_id': Scheme(shape=(), dtype=torch.int64)}
       edata_schemes={'edge_param': Scheme(shape=(1,), dtype=torch.float32)}),
 tensor(0))

In [32]:
test_loader

In [48]:
num_classes = {
    "": 2,
    "Class I (Retrotransposons)" : 2,
    "Class II (DNA transposons)": 10,
    "Class I (Retrotransposons)\tLTR Retrotransposon": 5,
    "Class I (Retrotransposons)\tNon-LTR Retrotransposon" : 7,
}
for root in HIERARCHY_ROOTS:

    if root == "":
        save_dir = Path("/Users/nad/mobiraph/data/n33_dotplots/models/root")
    else:
        save_dir = Path(f"/Users/nad/mobiraph/data/n33_dotplots/models/{root}")

    save_dir.mkdir(parents=True, exist_ok=True)

    save_path = save_dir / "gat_model.pt"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    _, sample_graph, _ = test_dataset[0]
    in_feats = sample_graph.ndata["feat"].shape[1]


    model = GATGraphClassifier(
        in_feats=in_feats,
        hidden_feats=64,
        num_classes=num_classes[root],
        num_heads=4,
        dropout=0.2,
        use_max_pool=False,
    ).to(device)

    model.load_state_dict(torch.load(save_path, map_location=device))
    model.to(device)
    model.eval()

    id2label = {v: k for k, v in test_dataset.label2id.items()}

    graph_ids, logits, preds, labels = predict_with_logits(model, test_loader, device)

    df_logits = pd.DataFrame(
        logits.numpy(),
        columns=[f"logit_{i}" for i in range(logits.shape[1])]
    )

    df_logits.insert(0, "graph_id", graph_ids)

    df_logits["pred_class_id"] = preds.numpy()
    df_logits["true_class_id"] = labels.numpy()
    df_logits["y_pred"] = df_logits["pred_class_id"].map(id2label)
    df_logits["y_true"] = df_logits["true_class_id"].map(id2label)

    df_logits.drop(columns=["pred_class_id", "true_class_id"], inplace=True)

    # переставим sample_id в начало
    cols = ["graph_id"] + [c for c in df_logits.columns if c != "graph_id"]
    df_logits = df_logits[cols]

    df_logits.rename(columns={'graph_id': 'name'}, inplace=True)
    df_logits.drop(columns=["y_true"], inplace=True)

    # сохраняем
    if root == "":
        df_logits.to_csv(f"/Users/nad/mobiraph/data/n35_train_results/root/gat.csv", index=False)
    else:
        df_logits.to_csv(f"/Users/nad/mobiraph/data/n35_train_results/{root}/gat.csv", index=False)

    print("Сохранено в test_logits.csv")
    df_logits.head()

Сохранено в test_logits.csv


In [31]:
torch.save(model.state_dict(), "/Users/nad/mobiraph/data/n33_dotplots/gat_model.pt")